# Age Group Detection — CNN (ResNet18 Transfer Learning)
## IN3060 Computer Vision Coursework

**References (cited at function level):**
- Week 7 tutorial: PyTorch basics, training loop, GPU, `torch.no_grad()`
- Week 8 tutorial: ResNet18 transfer learning, `train_model` function, `imshow`, `visualize_model`  
  *(Author: Chilamkurthy 2017, modified by Tarroni)*
- Week 9 tutorial: Custom `Dataset` class pattern (`CarvanaDataset`)
- He et al. (2016), *Deep Residual Learning for Image Recognition*, CVPR

> **Important:** Enable GPU in Colab before running:  
> *Runtime → Change runtime type → T4 GPU*

## 1. Notebook Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# TODO: set to the path of CW_Folder_UG inside your Google Drive (after 'My Drive/')
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'CW_Folder_UG'

GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print('CW folder contents:', os.listdir(GOOGLE_DRIVE_PATH))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import time

# PyTorch — Week 7 and 8 tutorials
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import models, transforms

from PIL import Image
from sklearn import metrics
from sklearn.utils.class_weight import compute_class_weight

%matplotlib inline

# Device setup — from Week 7 tutorial
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

CLASS_NAMES  = ['Child', 'Young', 'Middle-Aged', 'Senior']
DATASET_PATH = os.path.join(GOOGLE_DRIVE_PATH, 'CW_Dataset')
MODELS_PATH  = os.path.join(GOOGLE_DRIVE_PATH, 'Models')
os.makedirs(MODELS_PATH, exist_ok=True)
print('Setup complete.')

## 2. Custom PyTorch Dataset

Pattern directly adapted from the **Week 9 tutorial** `CarvanaDataset` class.
Reads labels from the txt file; images are loaded lazily in `__getitem__`.

In [ ]:
class AgeGroupDataset(Dataset):
    """
    PyTorch Dataset for Age Group Detection.
    Reference: Week 9 tutorial — CarvanaDataset class structure.
    """
    def __init__(self, img_dir, label_file, transform=None):
        with open(label_file, 'r') as f:
            self.labels = [int(line.strip().split(' ')[-1]) for line in f.readlines()]
        self.img_paths = sorted([
            os.path.join(img_dir, f)
            for f in os.listdir(img_dir) if f.endswith('.jpg')
        ])
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

## 3. Data Transforms and Augmentation

Directly adapted from the **Week 8 tutorial** `data_transforms` dictionary.

- **Training**: augmentation (random flip, colour jitter) reduces overfitting.
- **Val / Test**: only resize and normalise — no augmentation.

ImageNet mean/std normalisation is required because ResNet18 was pre-trained
on ImageNet with these statistics (from Week 8 tutorial).

In [ ]:
# ImageNet channel means and stds — from Week 8 tutorial
data_means = [0.485, 0.456, 0.406]
data_stds  = [0.229, 0.224, 0.225]

# Adapted from Week 8 tutorial data_transforms
# Stronger augmentation to combat overfitting on a small dataset
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((160, 160)),                        # resize larger first
        transforms.RandomCrop(128),                           # random crop to 128
        transforms.RandomHorizontalFlip(),                    # faces are symmetric
        transforms.RandomRotation(15),                        # slight rotation variation
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomGrayscale(p=0.1),                    # occasional greyscale
        transforms.ToTensor(),
        transforms.Normalize(data_means, data_stds)
    ]),
    'val': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(data_means, data_stds)
    ]),
}

## 4. Build Datasets and DataLoaders

In [ ]:
train_img_dir  = os.path.join(DATASET_PATH, 'train')
test_img_dir   = os.path.join(DATASET_PATH, 'test')
train_lbl_file = os.path.join(train_img_dir, 'train_labels.txt')
test_lbl_file  = os.path.join(test_img_dir,  'test_labels.txt')

# Full training dataset (with augmentation)
full_train_ds = AgeGroupDataset(train_img_dir, train_lbl_file,
                                 transform=data_transforms['train'])

# 80/20 split using random_split — from Week 9 tutorial
val_size   = int(0.2 * len(full_train_ds))
train_size = len(full_train_ds) - val_size
train_ds, val_ds_indices = random_split(
    full_train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Val split with NO augmentation — separate dataset object with val transforms
val_ds_base = AgeGroupDataset(train_img_dir, train_lbl_file,
                               transform=data_transforms['val'])

class IndexedSubset(Dataset):
    """Apply a dataset to only the specified indices."""
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx): return self.dataset[self.indices[idx]]

val_ds = IndexedSubset(val_ds_base, val_ds_indices.indices)

test_ds = AgeGroupDataset(test_img_dir, test_lbl_file,
                           transform=data_transforms['val'])

BATCH_SIZE = 32

# DataLoaders — from Week 7 and 8 tutorials
dataloaders = {
    'train': DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2),
    'val':   DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
}
dataset_sizes = {'train': len(train_ds), 'val': len(val_ds)}
test_loader   = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {dataset_sizes["train"]}  |  Val: {dataset_sizes["val"]}  |  Test: {len(test_ds)}')

# Visualise a batch — from Week 7 tutorial imshow helper
def imshow(inp, title=None):
    """Display a normalised tensor as an image. From Week 7 tutorial."""
    inp = inp.numpy().transpose((1, 2, 0))
    inp = np.asarray(data_stds) * inp + np.asarray(data_means)  # un-normalise
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title, fontsize=8)

first_inputs, first_labels = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(first_inputs[:8])
plt.figure(figsize=(14, 4))
imshow(out, title=[CLASS_NAMES[l] for l in first_labels[:8]])
plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Class Weights (Imbalance Mitigation)

In [ ]:
train_labels_list = [full_train_ds.labels[i] for i in train_ds.indices]
cw = compute_class_weight('balanced', classes=np.array([0, 1, 2, 3]),
                           y=train_labels_list)
class_weights_tensor = torch.tensor(cw, dtype=torch.float).to(device)
print(f'Class weights: {dict(enumerate(cw.round(3)))}')

## 6. Model — ResNet18 Transfer Learning

Directly adapted from the **Week 8 tutorial** ResNet18 transfer learning example
*(Author: Chilamkurthy 2017, modified by Tarroni)*.

Load ImageNet-pretrained ResNet18 and replace only the final fully-connected
layer with a 4-class linear layer. This reuses features learned on millions of
images (textures, edges, shapes) that transfer well to face-based classification.

Reference: Week 8 tutorial; He et al. (2016) ResNet.

In [ ]:
# Week 8 tutorial: model_ft = models.resnet18(weights='IMAGENET1K_V1')
model_ft = models.resnet18(weights='IMAGENET1K_V1')

# Replace final FC layer — identical to Week 8 tutorial
# Add Dropout before the linear layer to regularise against overfitting
num_ftrs    = model_ft.fc.in_features
model_ft.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_ftrs, len(CLASS_NAMES))
)
model_ft = model_ft.to(device)

# Weighted cross-entropy with label smoothing — reduces overconfident predictions
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

# ── Two-phase training strategy ───────────────────────────────────────────────
# Phase 1: freeze entire backbone, train only FC head (warms up the classifier).
# Phase 2: unfreeze only the last residual block (layer4) + FC for fine-tuning.
#   Keeping early layers frozen prevents destroying low-level ImageNet features
#   and greatly reduces overfitting compared to unfreezing all 11M parameters.

# ── Phase 1 setup (backbone fully frozen) ────────────────────────────────────
for param in model_ft.parameters():
    param.requires_grad = False
for param in model_ft.fc.parameters():
    param.requires_grad = True

optimizer_phase1 = optim.Adam(model_ft.fc.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_phase1 = lr_scheduler.StepLR(optimizer_phase1, step_size=3, gamma=0.5)

total_params     = sum(p.numel() for p in model_ft.parameters())
trainable_params = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f'Backbone      : ResNet18 (pretrained on ImageNet)')
print(f'Final FC      : {num_ftrs} → Dropout(0.5) → {len(CLASS_NAMES)} classes')
print(f'Total params  : {total_params:,}')
print(f'Phase 1 trains: {trainable_params:,} params (FC head only)')

## 7. `train_model` Function

Taken **directly** from the **Week 8 tutorial** `train_model` function
*(Author: Chilamkurthy 2017, modified by Tarroni)*.

Addition to the tutorial version: **early stopping** with configurable patience.

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=30, patience=7):
    """
    Train model for up to num_epochs, saving the best val-accuracy weights.
    Early stopping triggers if val accuracy does not improve for `patience` epochs.

    Source: Week 8 tutorial (Chilamkurthy 2017, modified by Tarroni).
    Addition: early stopping.
    """
    since = time.time()
    best_model_wts   = copy.deepcopy(model.state_dict())
    best_acc         = 0.0
    epochs_no_improv = 0

    train_losses, train_accs = [], []
    val_losses,   val_accs   = [], []

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 20)

        # Each epoch has a training and validation phase — Week 8 tutorial
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()   # set to training mode
            else:
                model.eval()    # set to evaluation mode

            running_loss     = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward (+ backward only in train phase) — Week 8 tutorial
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()   # update LR — Week 8 tutorial

            epoch_loss = running_loss     / dataset_sizes[phase]
            epoch_acc  = running_corrects.double() / dataset_sizes[phase]
            print(f'  {phase:5s}  Loss: {epoch_loss:.4f}   Acc: {epoch_acc:.4f}')

            if phase == 'train':
                train_losses.append(epoch_loss)
                train_accs.append(epoch_acc.item())
            else:
                val_losses.append(epoch_loss)
                val_accs.append(epoch_acc.item())
                # Keep best weights — Week 8 tutorial
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improv = 0
                else:
                    epochs_no_improv += 1

        if epochs_no_improv >= patience:
            print(f'\nEarly stopping: no improvement for {patience} epochs.')
            break
        print()

    elapsed = time.time() - since
    print(f'Training complete in {elapsed // 60:.0f}m {elapsed % 60:.0f}s')
    print(f'Best val accuracy: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)   # load best weights — Week 8 tutorial
    return model, train_losses, train_accs, val_losses, val_accs

## 8. Train

In [ ]:
# ── Phase 1: train FC head only (backbone frozen) ────────────────────────────
print('=' * 50)
print('PHASE 1 — FC head only (backbone frozen)')
print('=' * 50)
model_ft, tl1, ta1, vl1, va1 = train_model(
    model_ft, criterion, optimizer_phase1, scheduler_phase1,
    num_epochs=5, patience=5
)

# ── Phase 2: unfreeze layer4 + FC only, fine-tune with very small lr ─────────
print('\n' + '=' * 50)
print('PHASE 2 — layer4 + FC fine-tuning (rest frozen)')
print('=' * 50)

# Only unfreeze the last residual block and FC — keeps overfitting in check
for param in model_ft.parameters():
    param.requires_grad = False
for name, param in model_ft.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True

trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f'Phase 2 trains: {trainable:,} params (layer4 + FC)')

# Small lr to gently fine-tune without destroying pretrained features
optimizer_phase2 = optim.Adam(
    filter(lambda p: p.requires_grad, model_ft.parameters()),
    lr=5e-5, weight_decay=1e-4
)
scheduler_phase2 = lr_scheduler.StepLR(optimizer_phase2, step_size=7, gamma=0.5)

model_ft, tl2, ta2, vl2, va2 = train_model(
    model_ft, criterion, optimizer_phase2, scheduler_phase2,
    num_epochs=25, patience=10
)

# Combine curves from both phases for plotting
train_losses = tl1 + tl2
train_accs   = ta1 + ta2
val_losses   = vl1 + vl2
val_accs     = va1 + va2

## 9. Training Curves

Directly from **Week 8 tutorial** plotting style.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, 'b-o', label='Train')
ax1.plot(val_losses,   'r-o', label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)

ax2.plot([a * 100 for a in train_accs], 'b-o', label='Train')
ax2.plot([a * 100 for a in val_accs],   'r-o', label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True)

plt.suptitle('CNN (ResNet18) — Training and Validation Curves', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Test-Set Evaluation

Adapted from the **Week 7 tutorial** test loop (`torch.no_grad()` inference).

In [ ]:
model_ft.eval()
all_preds, all_labels = [], []
t0 = time.time()

with torch.no_grad():              # Week 7 tutorial
    for inputs, labels in test_loader:
        inputs  = inputs.to(device)
        outputs = model_ft(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

infer_time_cnn = (time.time() - t0) / len(all_labels) * 1000
test_acc_cnn   = metrics.accuracy_score(all_labels, all_preds)

print(f'Test accuracy   : {test_acc_cnn:.4f}')
print(f'Inference time  : {infer_time_cnn:.2f} ms / image')
print(metrics.classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

metrics.ConfusionMatrixDisplay.from_predictions(
    all_labels, all_preds, display_labels=CLASS_NAMES)
plt.title('Confusion Matrix — CNN (ResNet18)')
plt.tight_layout()
plt.show()

## 11. Qualitative Results

Adapted from **Week 8 tutorial** `visualize_model` function
*(Chilamkurthy / Tarroni)*.

In [ ]:
# Week 8 tutorial: visualize_model function
model_ft.eval()
images_shown = 0
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.ravel()

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs  = inputs.to(device)
        outputs = model_ft(inputs)
        _, preds = torch.max(outputs, 1)

        for j in range(inputs.size(0)):
            if images_shown >= 10:
                break
            ax  = axes[images_shown]
            inp = inputs[j].cpu().numpy().transpose((1, 2, 0))
            inp = np.asarray(data_stds) * inp + np.asarray(data_means)  # un-normalise
            inp = np.clip(inp, 0, 1)
            ax.imshow(inp)
            ax.set_title(
                f'GT: {CLASS_NAMES[labels[j]]}\nPred: {CLASS_NAMES[preds[j]]}',
                color='green' if preds[j] == labels[j] else 'red',
                fontsize=9
            )
            ax.set_axis_off()
            images_shown += 1

        if images_shown >= 10:
            break

plt.suptitle('Qualitative results — CNN ResNet18 (green=correct, red=wrong)', fontsize=11)
plt.tight_layout()
plt.show()

## 12. Save Model

In [ ]:
# torch.save — from Week 7 tutorial
model_path = os.path.join(MODELS_PATH, 'cnn_resnet18.pth')
torch.save(model_ft.state_dict(), model_path)
print(f'Model saved to: {model_path}')
print(f'CNN Test Accuracy: {test_acc_cnn:.4f}')